# LeetCode 857 — Minimum Cost to Hire K Workers bằng Python

- Link: https://leetcode.com/problems/minimum-cost-to-hire-k-workers/
- Độ khó: **Hard**
- Chủ đề: Tham lam, Sắp xếp theo tỉ lệ, Heap (max-heap), Chọn "đội trưởng" (enumerate the bottleneck)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def mincostToHireWorkers(self, quality: List[int], wage: List[int], k: int) -> float:
```

## 1. Giới thiệu & mục tiêu học

Có `n` công nhân, người thứ `i` có **chất lượng** `quality[i]` và **lương tối thiểu** `wage[i]`. Ta phải thuê **đúng `k`** người sao cho:

1. Mỗi người được trả **tỉ lệ thuận** với chất lượng của họ (cùng một "đơn giá" cho mọi người trong nhóm).
2. Mỗi người nhận ít nhất lương tối thiểu của mình.

Tìm tổng tiền nhỏ nhất.

Mấu chốt: nếu đơn giá là `r` thì người `i` nhận `r · quality[i] >= wage[i]` ⇔ `r >= wage[i] / quality[i]`. Vậy đơn giá của nhóm = **tỉ lệ lớn nhất** trong nhóm, tổng tiền = `max_ratio × Σ quality`. Bài trở thành: chọn `k` người cực tiểu `max(tỉ lệ) × tổng chất lượng` — kỹ thuật kinh điển **cố định phần tử "nút thắt" rồi tối ưu phần còn lại bằng heap**.

Sau notebook này bạn sẽ:

1. Biến đổi điều kiện đề bài thành công thức `cost = max_ratio · Σ quality`.
2. Viết brute force duyệt mọi tổ hợp `k` người.
3. Hiểu kỹ thuật **"đội trưởng"**: cố định người có tỉ lệ lớn nhất, chọn `k − 1` người chất lượng nhỏ nhất trong số người có tỉ lệ nhỏ hơn.
4. Tối ưu thành `O(n log n)` bằng sắp xếp theo tỉ lệ + **max-heap** giữ `k` chất lượng nhỏ nhất.
5. So sánh số thực với sai số cho phép.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho hai mảng `quality`, `wage` độ dài `n` và số `k`. Chọn đúng `k` công nhân và trả lương cho họ sao cho: (1) lương của mỗi người tỉ lệ với chất lượng của họ so với những người khác trong nhóm; (2) mỗi người nhận ít nhất `wage[i]`. Trả về tổng lương nhỏ nhất. Đáp án sai số trong `10⁻⁵` được chấp nhận.

**Ví dụ 1**

```text
Input: quality = [10,20,5], wage = [70,50,30], k = 2
Output: 105.00000
Giải thích: thuê người 0 (trả 70) và người 2 (trả 35).
```

**Ví dụ 2**

```text
Input: quality = [3,1,10,10,1], wage = [4,8,2,2,7], k = 3
Output: 30.66667
Giải thích: thuê người 0 (trả 4), người 2 và 3 (mỗi người 13.33333).
```

**Ràng buộc**

- `n == quality.length == wage.length`
- `1 <= k <= n <= 10⁴`
- `1 <= quality[i], wage[i] <= 10⁴`

In [1]:
from typing import List

EXAMPLES = [
    ([10, 20, 5], [70, 50, 30], 2, 105.0),
    ([3, 1, 10, 10, 1], [4, 8, 2, 2, 7], 3, 30.666666666666668),
    ([5], [10], 1, 10.0),
    ([1, 1, 1], [1, 2, 3], 3, 9.0),           # đơn giá = 3, tổng chất lượng 3
    ([4, 2], [8, 2], 1, 2.0),                 # k = 1: chọn người lương thấp nhất
]

def close(a, b, eps=1e-6):
    return abs(a - b) <= eps * max(1.0, abs(b))

for q, w, k, exp in EXAMPLES:
    print(q, w, k, "-> kỳ vọng %.5f" % exp)

[10, 20, 5] [70, 50, 30] 2 -> kỳ vọng 105.00000
[3, 1, 10, 10, 1] [4, 8, 2, 2, 7] 3 -> kỳ vọng 30.66667
[5] [10] 1 -> kỳ vọng 10.00000
[1, 1, 1] [1, 2, 3] 3 -> kỳ vọng 9.00000
[4, 2] [8, 2] 1 -> kỳ vọng 2.00000


## 3. Kiến thức nền

### 3.1. Tỉ lệ `wage / quality`

Tỉ lệ `ratio[i] = wage[i] / quality[i]` là **đơn giá tối thiểu** người `i` chấp nhận (tiền cho mỗi đơn vị chất lượng).

In [2]:
q, w = [10, 20, 5], [70, 50, 30]
print([wi / qi for qi, wi in zip(q, w)])

[7.0, 2.5, 6.0]


### 3.2. Sắp xếp theo khoá

`sorted(range(n), key=lambda i: ratio[i])` cho thứ tự chỉ số theo tỉ lệ tăng dần.

In [3]:
ratio = [wi / qi for qi, wi in zip(q, w)]
order = sorted(range(3), key=lambda i: ratio[i])
print(order, [ratio[i] for i in order])

[1, 2, 0] [2.5, 6.0, 7.0]


### 3.3. Max-heap trong Python

`heapq` là **min-heap**. Muốn max-heap: đẩy số **âm**. `-heap[0]` là phần tử lớn nhất.

In [4]:
import heapq
h = []
for x in [5, 1, 8, 3]:
    heapq.heappush(h, -x)
print("lớn nhất:", -h[0])
heapq.heappop(h)
print("sau khi bỏ lớn nhất, lớn nhất mới:", -h[0])

lớn nhất: 8
sau khi bỏ lớn nhất, lớn nhất mới: 5


### 3.4. Giữ `k` phần tử nhỏ nhất bằng max-heap kích thước `k`

Duyệt từng số, đẩy vào max-heap; nếu kích thước vượt `k` thì bỏ phần tử lớn nhất. Heap luôn chứa `k` số nhỏ nhất đã thấy; duy trì thêm biến `tổng`.

In [5]:
def k_smallest_stream(nums, k):
    h, s, out = [], 0, []
    for x in nums:
        heapq.heappush(h, -x); s += x
        if len(h) > k:
            s += heapq.heappop(h)          # pop trả về số âm của phần tử lớn nhất
        out.append((sorted(-v for v in h), s))
    return out

for state in k_smallest_stream([7, 2, 9, 1, 5], 2):
    print(state)

([7], 7)
([2, 7], 9)
([2, 7], 9)
([1, 2], 3)
([1, 2], 3)


### 3.5. `itertools.combinations` cho brute force

In [6]:
from itertools import combinations
print(list(combinations(range(4), 2)))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Biến đổi điều kiện

Gọi `r` là đơn giá chung: người `i` nhận `r · quality[i]`. Điều kiện (2): `r · quality[i] >= wage[i]` ⇔ `r >= ratio[i]`. Muốn tổng tiền `r · Σ quality` nhỏ nhất → chọn `r` nhỏ nhất có thể = `max(ratio[i] : i trong nhóm)`.

**→ cost(nhóm) = max ratio × Σ quality.**

### 4.2. Tính tay ví dụ 1

`ratio = [7, 2.5, 6]`. Các nhóm 2 người:

| nhóm | max ratio | Σ quality | cost |
|---|---|---|---|
| {0, 1} | 7 | 30 | 210 |
| {0, 2} | 7 | 15 | **105** |
| {1, 2} | 6 | 25 | 150 |

→ 105.

### 4.3. Hai lực kéo ngược nhau

Muốn `max ratio` nhỏ → chọn người "rẻ tính" (tỉ lệ thấp). Muốn `Σ quality` nhỏ → chọn người chất lượng thấp. Hai tiêu chí xung đột → không thể sắp theo một tiêu chí rồi lấy `k` người đầu. Cách gỡ: **cố định** người quyết định `max ratio` (gọi là "đội trưởng"), khi đó chỉ còn một tiêu chí.

In [7]:
q, w = [10, 20, 5], [70, 50, 30]
ratio = [wi / qi for qi, wi in zip(q, w)]
for grp in combinations(range(3), 2):
    mr = max(ratio[i] for i in grp); sq = sum(q[i] for i in grp)
    print(grp, "max ratio", mr, "Σq", sq, "cost", mr * sq)

(0, 1) max ratio 7.0 Σq 30 cost 210.0
(0, 2) max ratio 7.0 Σq 15 cost 105.0
(1, 2) max ratio 6.0 Σq 25 cost 150.0


### 4.4. Thêm ví dụ tính tay: ví dụ 2 theo từng "đội trưởng"

`quality = [3, 1, 10, 10, 1]`, `wage = [4, 8, 2, 2, 7]`, `k = 3` → `ratio = [1.333, 8, 0.2, 0.2, 7]`.

| đội trưởng c | ratio[c] | ứng viên (ratio ≤ ratio[c], khác c) | 2 chất lượng nhỏ nhất | Σq | cost |
|---|---|---|---|---|---|
| 0 | 1.333 | 2, 3 (q = 10, 10) | 10, 10 | 23 | **30.667** |
| 1 | 8 | 0, 2, 3, 4 | 1 (người 4), 3 (người 0) | 5 | 40 |
| 2 | 0.2 | 3 | chỉ 1 người — không đủ | — | — |
| 3 | 0.2 | 2 | không đủ | — | — |
| 4 | 7 | 0, 2, 3 | 3, 10 | 14 | 98 |

→ `30.667`. Nhận xét: đội trưởng có tỉ lệ cao (người 1) cho phép chọn người chất lượng thấp, nhưng đơn giá quá đắt; đội trưởng tỉ lệ thấp (người 0) buộc phải nhận người chất lượng cao nhưng đơn giá rẻ — ở đây rẻ hơn.

In [8]:
q2, w2, k2 = [3, 1, 10, 10, 1], [4, 8, 2, 2, 7], 3
r2 = [w / q for q, w in zip(q2, w2)]
for c in range(5):
    cand = sorted((q2[i], i) for i in range(5) if i != c and r2[i] <= r2[c])
    if len(cand) >= k2 - 1:
        pick = cand[:k2 - 1]
        sq = q2[c] + sum(q for q, _ in pick)
        print(f"đội trưởng {c}: chọn thêm {[i for _, i in pick]}, Σq={sq}, cost={r2[c] * sq:.3f}")
    else:
        print(f"đội trưởng {c}: không đủ ứng viên")

đội trưởng 0: chọn thêm [2, 3], Σq=23, cost=30.667
đội trưởng 1: chọn thêm [4, 0], Σq=5, cost=40.000
đội trưởng 2: không đủ ứng viên
đội trưởng 3: không đủ ứng viên
đội trưởng 4: chọn thêm [0, 2], Σq=14, cost=98.000


## 5. Cách 1 — Brute Force: duyệt mọi nhóm `k` người

In [9]:
def cost_brute(quality, wage, k):
    n = len(quality)
    ratio = [wage[i] / quality[i] for i in range(n)]
    best = float('inf')
    for grp in combinations(range(n), k):
        best = min(best, max(ratio[i] for i in grp) * sum(quality[i] for i in grp))
    return best

for q, w, k, exp in EXAMPLES:
    assert close(cost_brute(q, w, k), exp)
print("Brute force: OK")

Brute force: OK


**Độ phức tạp.** `C(n, k)` nhóm × `O(k)` → bùng nổ tổ hợp (`C(10⁴, 5000)` là con số khổng lồ). Chỉ dùng với `n <= 12`.

**Nút thắt.** Với **một đội trưởng cố định** (người có tỉ lệ lớn nhất trong nhóm), đơn giá đã biết; phần còn lại chỉ cần cực tiểu `Σ quality` — tức là lấy `k − 1` người **chất lượng nhỏ nhất** trong số những người có tỉ lệ ≤ đội trưởng. Không cần thử mọi tổ hợp!

## 6. Bước đột phá

**Insight 1 — Cố định đội trưởng.** Với mỗi `c`, giả sử `c` có tỉ lệ lớn nhất trong nhóm. Ứng viên đi cùng phải có `ratio <= ratio[c]`. Để cực tiểu `ratio[c] · (quality[c] + Σ quality khác)`, chọn `k − 1` người chất lượng **nhỏ nhất** trong số ứng viên. Đáp án = min theo `c`.

**Insight 2 — Sắp xếp theo tỉ lệ.** Sắp người theo `ratio` tăng dần. Khi duyệt tới người thứ `j` (làm đội trưởng), tập ứng viên chính là **mọi người đứng trước** (cộng chính `j`). Tập này **chỉ lớn dần** → duy trì `k` chất lượng nhỏ nhất bằng **max-heap** kích thước `k` và biến `sum_q`.

**Insight 3 — Không cần đảm bảo đội trưởng nằm trong heap.** Ở bước `j`, ta tính `ratio[j] · sum_q` với heap là `k` chất lượng nhỏ nhất trong `{0..j}`. Nếu người `j` bị loại khỏi heap (chất lượng quá lớn), giá trị này dùng đơn giá `ratio[j]` cao hơn cần thiết cho nhóm đó → chỉ **lớn hơn hoặc bằng** chi phí thật của nhóm, nên không làm sai đáp án min (nhóm đó đã được xét với đội trưởng đúng ở bước trước).

**Quy tắc nếu…thì:**
- *Nếu* heap có đúng `k` phần tử *thì* cập nhật `best = min(best, ratio[j] · sum_q)`.
- *Nếu* heap vượt `k` *thì* bỏ chất lượng lớn nhất.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử từng đội trưởng, sắp xếp ứng viên, `O(n² log n)`

In [10]:
def cost_captain(quality, wage, k):
    n = len(quality)
    ratio = [wage[i] / quality[i] for i in range(n)]
    best = float('inf')
    for c in range(n):
        others = sorted(quality[i] for i in range(n) if i != c and ratio[i] <= ratio[c])
        if len(others) >= k - 1:
            best = min(best, ratio[c] * (quality[c] + sum(others[:k - 1])))
    return best

for q, w, k, exp in EXAMPLES:
    assert close(cost_captain(q, w, k), exp)
print("Thử đội trưởng: OK")

Thử đội trưởng: OK


Lưu ý khi nhiều người có **cùng** tỉ lệ: điều kiện `ratio[i] <= ratio[c]` cho phép họ đi cùng nhau — đúng, vì đơn giá vẫn là `ratio[c]`.

Code "nộp" cho cách 2 (`O(n² log n)` — TLE với `n = 10⁴`, chỉ để tham khảo):

In [11]:
class SolutionCaptain:
    def mincostToHireWorkers(self, quality: List[int], wage: List[int], k: int) -> float:
        n = len(quality)
        ratio = [wage[i] / quality[i] for i in range(n)]
        best = float('inf')
        for c in range(n):
            others = sorted(quality[i] for i in range(n) if i != c and ratio[i] <= ratio[c])
            if len(others) >= k - 1:
                best = min(best, ratio[c] * (quality[c] + sum(others[:k - 1])))
        return best

for q, w, k, exp in EXAMPLES:
    assert close(SolutionCaptain().mincostToHireWorkers(q, w, k), exp)
print("SolutionCaptain: OK")

SolutionCaptain: OK


**Độ phức tạp cách 2:** `n` đội trưởng × sắp xếp `O(n log n)` → `O(n² log n)`; bộ nhớ `O(n)`.

### 7.2. Cách 3 — Đội trưởng + `heapq.nsmallest`, `O(n² log k)`

Thay sắp xếp toàn bộ bằng `heapq.nsmallest(k − 1, ...)`.

In [12]:
def cost_captain_nsmallest(quality, wage, k):
    n = len(quality)
    ratio = [wage[i] / quality[i] for i in range(n)]
    best = float('inf')
    for c in range(n):
        cand = [quality[i] for i in range(n) if i != c and ratio[i] <= ratio[c]]
        if len(cand) >= k - 1:
            best = min(best, ratio[c] * (quality[c] + sum(heapq.nsmallest(k - 1, cand))))
    return best

for q, w, k, exp in EXAMPLES:
    assert close(cost_captain_nsmallest(q, w, k), exp)
print("Đội trưởng + nsmallest: OK")

Đội trưởng + nsmallest: OK


Cải tiến nhỏ nhưng vẫn `O(n²)` — chưa đủ. Bước tiếp theo: dùng thứ tự theo tỉ lệ để **tái sử dụng** tập ứng viên giữa các đội trưởng.

### 7.3. Cách 4 — Sắp theo tỉ lệ + max-heap, `O(n log n)`

In [13]:
def cost_heap(quality, wage, k):
    workers = sorted((w / q, q) for q, w in zip(quality, wage))
    heap = []                 # max-heap (lưu số âm) các chất lượng đang chọn
    sum_q = 0
    best = float('inf')
    for r, q in workers:
        heapq.heappush(heap, -q)
        sum_q += q
        if len(heap) > k:
            sum_q += heapq.heappop(heap)     # bỏ chất lượng lớn nhất (pop trả về -q_max)
        if len(heap) == k:
            best = min(best, r * sum_q)
    return best

for q, w, k, exp in EXAMPLES:
    assert close(cost_heap(q, w, k), exp)
print("Sắp xếp + max-heap: OK")

Sắp xếp + max-heap: OK


Giải thích từng dòng:

- `workers` sắp theo tỉ lệ tăng dần → khi xử lý người `(r, q)`, mọi người trước đó có tỉ lệ ≤ `r`.
- Đẩy `q` vào heap, cộng vào `sum_q`.
- Nếu heap có `k + 1` phần tử, bỏ chất lượng **lớn nhất** (heap lưu số âm nên `heappop` trả về `−q_max`, cộng vào `sum_q` là trừ đi).
- Khi đủ `k` người, `r · sum_q` là chi phí khi lấy người hiện tại làm đội trưởng (hoặc cận trên của một nhóm đã xét — xem Insight 3).

Code nộp LeetCode cho cách 4:

In [14]:
class SolutionHeap:
    def mincostToHireWorkers(self, quality: List[int], wage: List[int], k: int) -> float:
        workers = sorted((w / q, q) for q, w in zip(quality, wage))
        heap, sum_q, best = [], 0, float('inf')
        for r, q in workers:
            heapq.heappush(heap, -q)
            sum_q += q
            if len(heap) > k:
                sum_q += heapq.heappop(heap)
            if len(heap) == k:
                best = min(best, r * sum_q)
        return best

for q, w, k, exp in EXAMPLES:
    assert close(SolutionHeap().mincostToHireWorkers(q, w, k), exp)
print("SolutionHeap: OK")

SolutionHeap: OK


**Độ phức tạp cách 4:** sắp xếp `O(n log n)` + `n` thao tác heap `O(log k)` → `O(n log n)`; bộ nhớ `O(n)`.

### 7.4. Cách 5 — Biến thể chính xác bằng phân số (tránh sai số)

Để so sánh tuyệt đối chính xác (ví dụ khi kiểm thử), dùng `fractions.Fraction`: `ratio = Fraction(w, q)`. Chậm hơn nhưng không có sai số làm tròn.

In [15]:
from fractions import Fraction

def cost_heap_exact(quality, wage, k):
    workers = sorted((Fraction(w, q), q) for q, w in zip(quality, wage))
    heap, sum_q, best = [], 0, None
    for r, q in workers:
        heapq.heappush(heap, -q)
        sum_q += q
        if len(heap) > k:
            sum_q += heapq.heappop(heap)
        if len(heap) == k:
            cur = r * sum_q
            if best is None or cur < best:
                best = cur
    return best

for q, w, k, exp in EXAMPLES:
    print(cost_heap_exact(q, w, k), "≈", float(cost_heap_exact(q, w, k)))
    assert close(float(cost_heap_exact(q, w, k)), exp)

105 ≈ 105.0
92/3 ≈ 30.666666666666668
10 ≈ 10.0
9 ≈ 9.0
2 ≈ 2.0


Code nộp LeetCode cho cách 5 (trả về `float` của kết quả chính xác):

In [16]:
class SolutionExact:
    def mincostToHireWorkers(self, quality: List[int], wage: List[int], k: int) -> float:
        return float(cost_heap_exact(quality, wage, k))

for q, w, k, exp in EXAMPLES:
    assert close(SolutionExact().mincostToHireWorkers(q, w, k), exp)
print("SolutionExact: OK")

SolutionExact: OK


**Độ phức tạp cách 5:** như cách 4 nhưng mỗi phép so sánh phân số đắt hơn (nhân số nguyên).

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 2: `quality = [3,1,10,10,1]`, `wage = [4,8,2,2,7]`, `k = 3`.

In [17]:
def trace_heap(quality, wage, k):
    workers = sorted((w / q, q, i) for i, (q, w) in enumerate(zip(quality, wage)))
    print("thứ tự theo tỉ lệ:", [(i, round(r, 4), q) for r, q, i in workers])
    heap, sum_q, best = [], 0, float('inf')
    for r, q, i in workers:
        heapq.heappush(heap, (-q, i))
        sum_q += q
        msg = f"thêm người {i} (ratio={r:.4f}, q={q})"
        if len(heap) > k:
            negq, j = heapq.heappop(heap)
            sum_q += negq
            msg += f"; bỏ người {j} (q={-negq})"
        if len(heap) == k:
            cost = r * sum_q
            best = min(best, cost)
            msg += f"; nhóm {sorted(j for _, j in heap)}, Σq={sum_q}, cost={r:.4f}×{sum_q}={cost:.5f}, best={best:.5f}"
        print(msg)
    return best

trace_heap([3, 1, 10, 10, 1], [4, 8, 2, 2, 7], 3)

thứ tự theo tỉ lệ: [(2, 0.2, 10), (3, 0.2, 10), (0, 1.3333, 3), (4, 7.0, 1), (1, 8.0, 1)]
thêm người 2 (ratio=0.2000, q=10)
thêm người 3 (ratio=0.2000, q=10)
thêm người 0 (ratio=1.3333, q=3); nhóm [0, 2, 3], Σq=23, cost=1.3333×23=30.66667, best=30.66667
thêm người 4 (ratio=7.0000, q=1); bỏ người 2 (q=10); nhóm [0, 3, 4], Σq=14, cost=7.0000×14=98.00000, best=30.66667
thêm người 1 (ratio=8.0000, q=1); bỏ người 3 (q=10); nhóm [0, 1, 4], Σq=5, cost=8.0000×5=40.00000, best=30.66667


30.666666666666664

Bảng trạng thái:

| bước | người (ratio, q) | heap (chỉ số) | Σq | cost | best |
|---|---|---|---|---|---|
| 1 | 2 (0.2, 10) | {2} | 10 | — | ∞ |
| 2 | 3 (0.2, 10) | {2, 3} | 20 | — | ∞ |
| 3 | 0 (1.333, 3) | {0, 2, 3} | 23 | 30.667 | 30.667 |
| 4 | 4 (7, 1) | bỏ một người q=10 → {0, 4, ·} | 14 | 98 | 30.667 |
| 5 | 1 (8, 1) | bỏ người q=10 còn lại → {0, 1, 4} | 5 | 40 | 30.667 |

Đáp án `30.66667`: nhóm `{0, 2, 3}` với đơn giá `4/3`.

## 9. Code nộp LeetCode

Chọn cách 4 (sắp theo tỉ lệ + max-heap): `O(n log n)`.

In [18]:
import heapq
from typing import List

class Solution:
    def mincostToHireWorkers(self, quality: List[int], wage: List[int], k: int) -> float:
        # Sắp công nhân theo đơn giá tối thiểu wage/quality tăng dần
        workers = sorted((w / q, q) for q, w in zip(quality, wage))
        heap = []             # max-heap (số âm) giữ k chất lượng nhỏ nhất
        sum_q = 0
        best = float('inf')
        for ratio, q in workers:
            heapq.heappush(heap, -q)
            sum_q += q
            if len(heap) > k:
                sum_q += heapq.heappop(heap)       # bỏ chất lượng lớn nhất
            if len(heap) == k:
                best = min(best, ratio * sum_q)    # người hiện tại quyết định đơn giá
        return best

for q, w, k, exp in EXAMPLES:
    got = Solution().mincostToHireWorkers(q, w, k)
    assert close(got, exp)
    print("%.5f" % got)

105.00000
30.66667
10.00000
9.00000
2.00000


**Giải thích từng dòng của `Solution`:**

| dòng | ý nghĩa |
|---|---|
| `workers = sorted((w / q, q) ...)` | sắp theo đơn giá tối thiểu tăng dần |
| `heap = []` | max-heap (lưu `−q`) chứa tối đa `k` chất lượng nhỏ nhất đã gặp |
| `sum_q` | tổng chất lượng trong heap |
| `heappush(heap, -q)` | người hiện tại là ứng viên |
| `if len(heap) > k: sum_q += heappop(heap)` | bỏ người chất lượng lớn nhất (pop trả về `−q_max`) |
| `if len(heap) == k: best = min(best, ratio * sum_q)` | người hiện tại có tỉ lệ lớn nhất trong số đã xét → quyết định đơn giá |

Vì sao trả về `float` là đủ? Đề chấp nhận sai số `10⁻⁵`. Đơn giá `ratio ≤ 10⁴` và tổng chất lượng `Σq ≤ 10⁴ · 10⁴ = 10⁸`, nên chi phí ≤ `10¹²`; `float` có khoảng 15–16 chữ số có nghĩa (sai số tương đối cỡ `10⁻¹⁶`) — hoàn toàn an toàn.

In [19]:
from fractions import Fraction
import random
rnd_chk = random.Random(1)
qq = [rnd_chk.randint(1, 10 ** 4) for _ in range(2000)]
ww = [rnd_chk.randint(1, 10 ** 4) for _ in range(2000)]
f_val = Solution().mincostToHireWorkers(qq, ww, 700)
e_val = cost_heap_exact(qq, ww, 700)
print(f"float: {f_val:.6f} | chính xác: {float(e_val):.6f} | sai số tương đối: {abs(f_val - float(e_val)) / float(e_val):.2e}")

float: 3158648.737035 | chính xác: 3158648.737035 | sai số tương đối: 1.47e-16


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Với nhóm `G` cố định, chi phí nhỏ nhất là `cost(G) = max_{i∈G} ratio[i] · Σ_{i∈G} quality[i]`. (Mục 4.1: đơn giá chung `r` phải ≥ mọi `ratio[i]`; chi phí `r·Σq` tăng theo `r`.)

**Bổ đề 2 (đội trưởng).** Gọi `c(G)` = người có tỉ lệ lớn nhất trong `G` (hoà thì lấy người đứng **sau** trong thứ tự sắp xếp). Khi thuật toán xử lý `c(G)`, heap chứa `k` chất lượng nhỏ nhất trong tập `P` = mọi người đã xử lý (tỉ lệ ≤ `ratio[c(G)]`). Vì `G ⊆ P` và `|G| = k`, nên `sum_q <= Σ_{i∈G} quality[i]`. Do đó giá trị thuật toán tính tại bước này `ratio[c(G)] · sum_q <= cost(G)`.

**Bổ đề 3 (không đánh giá thấp).** Mọi giá trị thuật toán tính đều là chi phí thật của **một** nhóm hợp lệ hoặc lớn hơn: tại bước của người `j`, heap là `k` người `H ⊆ P`, mọi người trong `H` có tỉ lệ ≤ `ratio[j]`, nên `ratio[j] · Σ_H q >= max_{H} ratio · Σ_H q = cost(H)` — tức ≥ chi phí tối ưu.

Kết hợp: `best <= cost(G*)` (bổ đề 2 với nhóm tối ưu `G*`) và `best >= cost tối ưu` (bổ đề 3) ⇒ `best` = chi phí tối ưu. ∎

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tổ hợp | `O(C(n,k) · k)` | `O(k)` |
| 2. Đội trưởng + sort | `O(n² log n)` | `O(n)` |
| 3. Đội trưởng + nsmallest | `O(n² log k)` | `O(n)` |
| 4. Sắp xếp + max-heap | `O(n log n)` | `O(n)` |
| 5. Như 4 với `Fraction` | `O(n log n)` (hằng số lớn) | `O(n)` |

In [20]:
import random, time

rnd = random.Random(857)
n = 10 ** 4
Q = [rnd.randint(1, 10 ** 4) for _ in range(n)]
Wg = [rnd.randint(1, 10 ** 4) for _ in range(n)]
for name, fn in [("heap", cost_heap), ("exact", lambda a, b, c: float(cost_heap_exact(a, b, c))),
                 ("Solution", lambda a, b, c: Solution().mincostToHireWorkers(a, b, c))]:
    t0 = time.perf_counter()
    res = fn(Q, Wg, 2500)
    print(f"{name:9}: {res:.5f}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")
t0 = time.perf_counter(); cost_captain(Q[:1500], Wg[:1500], 300); print(f"đội trưởng + sort n=1500: {(time.perf_counter() - t0) * 1000:.0f} ms")

heap     : 8301693.25648  (12 ms)


exact    : 8301693.25648  (142 ms)
Solution : 8301693.25648  (9 ms)


đội trưởng + sort n=1500: 276 ms


## 12. Bộ test

In [21]:
methods = {
    "brute": cost_brute,
    "captain": cost_captain,
    "captain_nsmallest": cost_captain_nsmallest,
    "heap": cost_heap,
    "exact": lambda a, b, c: float(cost_heap_exact(a, b, c)),
    "SolutionCaptain": lambda a, b, c: SolutionCaptain().mincostToHireWorkers(a, b, c),
    "SolutionHeap": lambda a, b, c: SolutionHeap().mincostToHireWorkers(a, b, c),
    "SolutionExact": lambda a, b, c: SolutionExact().mincostToHireWorkers(a, b, c),
    "Solution": lambda a, b, c: Solution().mincostToHireWorkers(a, b, c),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([2, 2, 2], [4, 4, 4], 2, 8.0),                 # mọi tỉ lệ bằng nhau
    ([1, 100], [1, 100], 2, 101.0),
    ([1, 2, 3, 4], [4, 3, 2, 1], 4, 40.0),          # k = n: đơn giá = max ratio = 4
]
for name, fn in methods.items():
    for q, w, k, exp in edge:
        assert close(fn(q, w, k), exp), (name, q, w, k, fn(q, w, k), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force (so sánh chính xác bằng Fraction)
def cost_brute_exact(quality, wage, k):
    n = len(quality)
    return min(max(Fraction(wage[i], quality[i]) for i in g) * sum(quality[i] for i in g)
               for g in combinations(range(n), k))

for seed in range(500):
    r_ = random.Random(seed)
    n = r_.randint(1, 9)
    k = r_.randint(1, n)
    q = [r_.randint(1, 20) for _ in range(n)]
    w = [r_.randint(1, 20) for _ in range(n)]
    ref = cost_brute_exact(q, w, k)
    assert cost_heap_exact(q, w, k) == ref, (q, w, k)
    for name, fn in methods.items():
        assert close(fn(q, w, k), float(ref)), (name, q, w, k)
print("500 bộ ngẫu nhiên: khớp brute force")

# 12.3 Dữ liệu vừa: cách O(n log n) khớp cách O(n^2)
for seed in range(10):
    r_ = random.Random(700 + seed)
    n = 300; k = r_.randint(1, n)
    q = [r_.randint(1, 10 ** 4) for _ in range(n)]
    w = [r_.randint(1, 10 ** 4) for _ in range(n)]
    assert close(cost_heap(q, w, k), cost_captain(q, w, k))
print("n = 300: heap khớp đội trưởng")

Ví dụ + edge cases: OK
500 bộ ngẫu nhiên: khớp brute force


n = 300: heap khớp đội trưởng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Sắp theo lương (hoặc chất lượng) rồi lấy `k` người đầu

**Code sai:**

In [22]:
def wrong_sort_by_wage(quality, wage, k):
    idx = sorted(range(len(wage)), key=lambda i: wage[i])[:k]       # SAI: một tiêu chí
    return max(wage[i] / quality[i] for i in idx) * sum(quality[i] for i in idx)

q, w, k, exp = EXAMPLES[0]
print("sai:", wrong_sort_by_wage(q, w, k), " đúng:", Solution().mincostToHireWorkers(q, w, k))

sai: 150.0  đúng: 105.0


**Vì sao sai:** ở ví dụ 1, hai người lương thấp nhất là người 2 (30) và người 1 (50), nhưng người 1 có chất lượng 20 → phải trả theo đơn giá `6` của người 2: `6 × 25 = 150`. Nhóm tối ưu `{0, 2}` có lương tối thiểu cao hơn nhưng tổng chất lượng nhỏ: `7 × 15 = 105`. Phải cân bằng **tỉ lệ** và **tổng chất lượng**.

### Lỗi 2 — Lấy tổng lương tối thiểu `Σ wage` của nhóm

**Code sai:**

In [23]:
def wrong_sum_wage(quality, wage, k):
    best = float('inf')
    for g in combinations(range(len(wage)), k):
        best = min(best, sum(wage[i] for i in g))                    # SAI: bỏ qua ràng buộc tỉ lệ
    return best

print("sai:", wrong_sum_wage([10, 20, 5], [70, 50, 30], 2), " đúng:", Solution().mincostToHireWorkers([10, 20, 5], [70, 50, 30], 2))

sai: 80  đúng: 105.0


**Vì sao sai:** lương phải **tỉ lệ với chất lượng**. Nhóm `{1, 2}`: trả 50 và 30 không tỉ lệ với `20 : 5`. Phải trả theo đơn giá chung `max ratio` → người 1 nhận `6 · 20 = 120`.

### Lỗi 3 — Dùng min-heap thay vì max-heap (bỏ nhầm chất lượng **nhỏ nhất**)

**Code sai:**

In [24]:
def wrong_min_heap(quality, wage, k):
    workers = sorted((w / q, q) for q, w in zip(quality, wage))
    heap, sum_q, best = [], 0, float('inf')
    for r, q in workers:
        heapq.heappush(heap, q)                                      # SAI: min-heap
        sum_q += q
        if len(heap) > k:
            sum_q -= heapq.heappop(heap)                             # bỏ chất lượng NHỎ nhất
        if len(heap) == k:
            best = min(best, r * sum_q)
    return best

q, w, k, exp = EXAMPLES[0]
print("sai:", wrong_min_heap(q, w, k), " đúng:", Solution().mincostToHireWorkers(q, w, k))

sai: 150.0  đúng: 105.0


**Vì sao sai:** ta muốn **giữ** các chất lượng nhỏ (để `Σq` nhỏ) và **bỏ** chất lượng lớn nhất. Python `heapq` là min-heap → lưu `−q`.

### Lỗi 4 — Tính `ratio` bằng chia nguyên

`w // q` làm tròn xuống → đơn giá sai (ví dụ `4 // 3 = 1` thay vì `1.333`). Dùng `/` (số thực) hoặc `Fraction`.

In [25]:
print("chia nguyên:", 4 // 3, "| chia thực:", 4 / 3)

chia nguyên: 1 | chia thực: 1.3333333333333333


## 14. Mẹo & nhận dạng pattern

- **"Chi phí = max(A) × sum(B)"** hoặc **"min(A) × sum(B)"** trên nhóm `k` phần tử → **sắp theo A**, duyệt phần tử quyết định max/min, dùng **heap kích thước k** để giữ `k` giá trị B tốt nhất.
- **Kỹ thuật "đội trưởng" / nút thắt:** cố định phần tử đạt cực trị của một tiêu chí → bài còn một tiêu chí.
- **Max-heap trong Python** = đẩy số âm.
- Bài cùng khuôn: **2542 Maximum Subsequence Score** (`min(nums2) × sum(nums1)`), **1383 Maximum Performance of a Team**, **502 IPO**, **630 Course Schedule III**.

## 15. Biến thể

1. **Trả về danh sách người được thuê**: lưu chỉ số trong heap, ghi lại nhóm khi cập nhật `best`.
2. **Thuê tối đa `k` người với ngân sách `B`, tối đa số người**: sắp theo tỉ lệ, với mỗi đội trưởng tìm số người lớn nhất có `ratio · Σq ≤ B` (giữ các chất lượng nhỏ nhất).
3. **Thuê ít nhất `k` người**: luôn lấy đúng `k` là tối ưu (thêm người chỉ tăng chi phí) → như bài gốc.
4. **Hiệu suất đội = min(efficiency) × Σ speed, tối đa hoá** (1383): sắp theo efficiency giảm dần, min-heap giữ `k` speed lớn nhất.
5. **Có thêm phí cố định mỗi người**: `cost = r·Σq + Σ fee` — không còn tách được gọn, cần cách khác (ví dụ thử đội trưởng và chọn theo `r·q + fee`).

Minh hoạ biến thể 1 và 4:

In [26]:
def hire_group(quality, wage, k):
    workers = sorted((w / q, q, i) for i, (q, w) in enumerate(zip(quality, wage)))
    heap, sum_q, best, group = [], 0, float('inf'), None
    for r, q, i in workers:
        heapq.heappush(heap, (-q, i)); sum_q += q
        if len(heap) > k:
            sum_q += heapq.heappop(heap)[0]
        if len(heap) == k and r * sum_q < best:
            best, group = r * sum_q, sorted(j for _, j in heap)
    return best, group

def max_performance(speed, efficiency, k):
    eng = sorted(zip(efficiency, speed), reverse=True)
    heap, s, best = [], 0, 0
    for e, sp in eng:
        heapq.heappush(heap, sp); s += sp
        if len(heap) > k:
            s -= heapq.heappop(heap)
        best = max(best, s * e)
    return best

print(hire_group([3, 1, 10, 10, 1], [4, 8, 2, 2, 7], 3))
print(max_performance([2, 10, 3, 1, 5, 8], [5, 4, 3, 9, 7, 2], 2))   # 60 (LeetCode 1383 ví dụ 1)

(30.666666666666664, [0, 2, 3])
60


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tổ hợp | `max ratio × Σq` từng nhóm | `O(C(n,k)·k)` | `O(k)` | đối chiếu, `n ≤ 12` |
| 2. Đội trưởng + sort | cố định max ratio | `O(n² log n)` | `O(n)` | hiểu ý tưởng |
| 3. Đội trưởng + nsmallest | như 2, `k−1` nhỏ nhất | `O(n² log k)` | `O(n)` | `n ≤ 2000` |
| 4. Sắp xếp + max-heap | tập ứng viên lớn dần | `O(n log n)` | `O(n)` | **nộp bài** |
| 5. Phân số chính xác | như 4 với `Fraction` | `O(n log n)` | `O(n)` | kiểm thử chính xác |

## 17. Bài tập

**Bài 1 (đoán output).** `quality = [1, 1]`, `wage = [5, 7]`, `k = 2`? `k = 1`?

**Bài 2 (tính tay).** Lập bảng trace cách 4 cho `quality = [4, 5, 2]`, `wage = [8, 10, 5]`, `k = 2`.

**Bài 3 (điền TODO).** Viết `max_hired_with_budget(quality, wage, B)` — số người nhiều nhất có thể thuê với tổng chi phí ≤ `B` (biến thể 2).

**Bài 4 (tự viết lại).** Viết lại `Solution` không dùng số thực: lưu tỉ lệ dạng cặp `(w, q)` và so sánh bằng nhân chéo; trả `float` ở cuối.

**Bài 5 (mini challenge).** LeetCode 2542: cho `nums1`, `nums2`, `k`, tối đa hoá `sum(nums1 trên nhóm) × min(nums2 trên nhóm)`. Cài và đối chiếu brute force.

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def max_hired_with_budget_todo(quality, wage, B):
    # TODO: sắp theo ratio; với mỗi đội trưởng, giữ max-heap chất lượng; bỏ lớn nhất khi ratio*sum_q > B
    pass

# Bài 4 — TODO
def solution_no_float_todo(quality, wage, k):
    # TODO: sort bằng functools.cmp_to_key với w1*q2 so với w2*q1; best lưu dạng (tử, mẫu)
    pass

# Bài 5 — TODO
def max_score_todo(nums1, nums2, k):
    # TODO: sắp theo nums2 giảm dần; min-heap giữ k phần tử nums1 lớn nhất
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tỉ lệ `[5, 7]`.
- Mức 2: `k = 2`: đơn giá 7, `Σq = 2`.
- Mức 3: `14`; `k = 1`: chọn người lương thấp nhất → `5`.

**Bài 2.**
- Mức 1: tỉ lệ `[2, 2, 2.5]`.
- Mức 2: sau 2 người đầu: `Σq = 9`, cost `18`.
- Mức 3: thêm người 2 (q=2): bỏ q=5 → `Σq = 6`, cost `2.5 × 6 = 15` → đáp án `15`.

**Bài 3.**
- Mức 1: số người tối đa với đội trưởng `c` = số chất lượng nhỏ nhất (trong số tỉ lệ ≤) có tổng ≤ `B / ratio[c]`.
- Mức 2: heap không giới hạn `k`; khi `ratio·sum_q > B`, bỏ chất lượng lớn nhất.
- Mức 3: kích thước heap sau khi thoả là ứng viên; lấy max (heap chỉ co khi cần — vẫn đúng vì ratio tăng dần).

**Bài 4.**
- Mức 1: so sánh `w1/q1 < w2/q2` ⇔ `w1·q2 < w2·q1`.
- Mức 2: chi phí `= w_c · Σq / q_c` → so sánh hai phân số bằng nhân chéo.
- Mức 3: `float(tử) / mẫu` ở cuối.

**Bài 5.**
- Mức 1: cố định phần tử có `nums2` nhỏ nhất trong nhóm.
- Mức 2: sắp `nums2` giảm dần; heap giữ `k` giá trị `nums1` lớn nhất.
- Mức 3: khi heap đủ `k`, cập nhật `sum × nums2[hiện tại]`.

## 19. Đáp án tham khảo

In [28]:
from functools import cmp_to_key

# Bài 1
assert close(Solution().mincostToHireWorkers([1, 1], [5, 7], 2), 14.0)
assert close(Solution().mincostToHireWorkers([1, 1], [5, 7], 1), 5.0)

# Bài 2
assert close(Solution().mincostToHireWorkers([4, 5, 2], [8, 10, 5], 2), 15.0)

# Bài 3
def max_hired_with_budget(quality, wage, B):
    workers = sorted((w / q, q) for q, w in zip(quality, wage))
    heap, sum_q, best = [], 0, 0
    for r, q in workers:
        heapq.heappush(heap, -q); sum_q += q
        while heap and r * sum_q > B + 1e-9:
            sum_q += heapq.heappop(heap)
        best = max(best, len(heap))
    return best

def max_hired_brute(quality, wage, B):
    n = len(quality)
    for k in range(n, 0, -1):
        if cost_brute(quality, wage, k) <= B + 1e-9:
            return k
    return 0

for seed in range(200):
    r_ = random.Random(seed)
    n = r_.randint(1, 8)
    q = [r_.randint(1, 10) for _ in range(n)]; w = [r_.randint(1, 10) for _ in range(n)]
    B = r_.randint(1, 80)
    assert max_hired_with_budget(q, w, B) == max_hired_brute(q, w, B), (q, w, B)

# Bài 4
def solution_no_float(quality, wage, k):
    def cmp(a, b):                                   # a, b = (w, q)
        return (a[0] * b[1] > b[0] * a[1]) - (a[0] * b[1] < b[0] * a[1])
    workers = sorted(zip(wage, quality), key=cmp_to_key(cmp))
    heap, sum_q, best = [], 0, None                  # best = (tử, mẫu)
    for w, q in workers:
        heapq.heappush(heap, -q); sum_q += q
        if len(heap) > k:
            sum_q += heapq.heappop(heap)
        if len(heap) == k:
            num, den = w * sum_q, q                  # cost = w/q * sum_q
            if best is None or num * best[1] < best[0] * den:
                best = (num, den)
    return best[0] / best[1]

for seed in range(200):
    r_ = random.Random(seed)
    n = r_.randint(1, 9); k = r_.randint(1, n)
    q = [r_.randint(1, 30) for _ in range(n)]; w = [r_.randint(1, 30) for _ in range(n)]
    assert close(solution_no_float(q, w, k), Solution().mincostToHireWorkers(q, w, k))

# Bài 5
def max_score(nums1, nums2, k):
    pairs = sorted(zip(nums2, nums1), reverse=True)
    heap, s, best = [], 0, 0
    for b, a in pairs:
        heapq.heappush(heap, a); s += a
        if len(heap) > k:
            s -= heapq.heappop(heap)
        if len(heap) == k:
            best = max(best, s * b)
    return best

def max_score_brute(nums1, nums2, k):
    return max(sum(nums1[i] for i in g) * min(nums2[i] for i in g) for g in combinations(range(len(nums1)), k))

assert max_score([1, 3, 3, 2], [2, 1, 3, 4], 3) == 12
for seed in range(200):
    r_ = random.Random(seed)
    n = r_.randint(1, 8); k = r_.randint(1, n)
    a = [r_.randint(0, 10) for _ in range(n)]; b = [r_.randint(0, 10) for _ in range(n)]
    assert max_score(a, b, k) == max_score_brute(a, b, k)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (sắp theo tiêu chí max + heap k phần tử):**

```python
workers = sorted((w / q, q) for q, w in zip(quality, wage))
heap, sum_q, best = [], 0, float('inf')
for r, q in workers:
    heapq.heappush(heap, -q); sum_q += q
    if len(heap) > k: sum_q += heapq.heappop(heap)
    if len(heap) == k: best = min(best, r * sum_q)
return best
```

**Checklist trước khi submit:**

- [ ] Chi phí nhóm = `max(wage/quality) × Σ quality`.
- [ ] Sắp theo **tỉ lệ** tăng dần.
- [ ] Heap là **max-heap** (đẩy `−q`), bỏ chất lượng **lớn nhất**.
- [ ] Chỉ cập nhật khi heap có **đúng `k`** phần tử.
- [ ] Dùng chia **thực** `/`, không `//`.